# 03 — Relations, evidence, provenance, and biological questions

This lesson separates graph assertions from their source records, performs bounded joins by typed edge identity, and asks a concrete TP53 question. Every result is interpreted as represented provenance-aware association—not causality, efficacy, safety, or a completeness claim.


In [ ]:
from __future__ import annotations
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from manage_db.public_notebooks import (
    PUBLIC_KG_ROOT,
    build_public_fixture,
    parquet_catalog,
    read_bounded_parquet,
)

MODE = os.environ.get("JOUVENCE_DATA_MODE", "fixture")
BILLING_PROJECT = os.environ.get("JOUVENCE_BILLING_PROJECT")
CACHE = Path(os.environ.get("JOUVENCE_NOTEBOOK_CACHE", REPO_ROOT / "artifacts" / "cache" / "public-notebooks"))
CACHE.mkdir(parents=True, exist_ok=True)
KG_ROOT = build_public_fixture(CACHE / "kg-fixture") if MODE == "fixture" else PUBLIC_KG_ROOT
print({"mode": MODE, "kg_root": str(KG_ROOT), "bounded": True})


## Read assertions separately from evidence

An edge table contains one deduplicated assertion per typed endpoint pair and relation. Its evidence table contains one or more source records with predicates, scores, studies, assays, releases, and provenance. The two surfaces have different row identities and cardinalities.


In [ ]:
# Read assertions separately from evidence: run the bounded example and inspect its typed output.
root = str(KG_ROOT).rstrip("/")
edges = read_bounded_parquet(f"{root}/edges/disease_associated_gene.parquet", limit=20, billing_project=BILLING_PROJECT)
evidence = read_bounded_parquet(f"{root}/evidence/disease_associated_gene.parquet", limit=50, billing_project=BILLING_PROJECT)
print({"assertions": len(edges), "evidence_rows": len(evidence)})
display(edges.head())


### Interpretation

A relation name should express the accepted source-native biological assertion. Assay, source predicate, score, and context refine evidence; they should not proliferate relation names or silently redefine endpoint meaning.


In [ ]:
evidence_columns = [column for column in ["edge_key", "source", "source_dataset", "source_record_id", "predicate", "evidence_score", "release"] if column in evidence]
display(evidence[evidence_columns].head(10))


### Checkpoint

Compare row identities before joining. Evidence multiplicity can exceed assertion count without creating duplicate topology.


## Construct and validate typed edge identity

The portable join key is relation plus both stable IDs and both endpoint types. Some tables also materialize a deterministic edge key. Row order is never a valid join contract because writers, filters, and Parquet row groups may reorder data.


In [ ]:
# Construct and validate typed edge identity: run the bounded example and inspect its typed output.
EDGE_IDENTITY = ["relation", "x_id", "x_type", "y_id", "y_type"]
assert set(EDGE_IDENTITY).issubset(edges) and set(EDGE_IDENTITY).issubset(evidence)
print(edges[EDGE_IDENTITY].drop_duplicates().to_string(index=False))


### Interpretation

Typed endpoints prevent an identifier string from being interpreted in the wrong namespace. Canonical human gene endpoints use ENSG IDs; aliases remain lookup or provenance fields rather than parallel canonical nodes.


In [ ]:
assert not edges.duplicated(EDGE_IDENTITY).any()
multiplicity = evidence.groupby(EDGE_IDENTITY, dropna=False).size().rename("evidence_rows").reset_index()
display(multiplicity.sort_values("evidence_rows", ascending=False))


### Checkpoint

A duplicate edge identity is a topology error; multiple evidence rows for one identity are expected when distinct source records support the assertion.


## Join bounded prefixes without claiming completeness

The reusable helper reads selected columns from bounded prefixes of one edge and evidence object and validates a one-to-many merge. It is designed for inspection, not full evidence parity or absence testing.


In [ ]:
# Join bounded prefixes without claiming completeness: run the bounded example and inspect its typed output.
from manage_db.public_notebooks import bounded_edge_evidence_join
joined = bounded_edge_evidence_join(KG_ROOT, "disease_associated_gene", edge_limit=10, evidence_limit=50, billing_project=BILLING_PROJECT)
display(joined.head(10))


### Interpretation

A bounded join may omit evidence that lies outside the selected prefix. Therefore zero joined rows means only that the bounded windows did not overlap; it does not prove that canonical evidence is absent.


In [ ]:
join_summary = joined.groupby("source_assertion", dropna=False).agg(assertion_evidence_pairs=("source_record_id", "size"), evidence_sources=("source_evidence", "nunique"))
display(join_summary)


### Checkpoint

Use full parity audits only in reviewed worker workflows. Public notebooks remain laptop-safe and never expand scope to settle an absence question.


## Ask a bounded TP53–disease question

We query the fixture for diseases attached to TP53 (`ENSG00000141510`), join disease labels, and summarize represented evidence. Exact stable IDs make the question reproducible despite ambiguous gene symbols.


In [ ]:
# Ask a bounded TP53–disease question: run the bounded example and inspect its typed output.
from manage_db.public_notebooks import diseases_with_gene_evidence
if MODE == "fixture":
    answer = diseases_with_gene_evidence(KG_ROOT, "ENSG00000141510", limit=20)
else:
    answer = pd.DataFrame()
    print("Live helper requires an explicitly staged bounded local subset.")
display(answer)


### Interpretation

The returned source and score describe records in this selected graph. They are not calibrated probabilities of disease causation, target validity, clinical success, or patient benefit.


In [ ]:
if not answer.empty:
    answer_plot = answer.sort_values("max_evidence_score")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.barh(answer_plot["disease_name"], answer_plot["max_evidence_score"], color="#0072B2")
    ax.set(xlabel="maximum source-specific fixture score", title="Represented TP53 associations")
    fig.tight_layout(); plt.show()


### Checkpoint

Treat the chart as a transparent view of selected source scores. Do not rank therapies or patients from this association plot.


## Distinguish observed assertions from controlled inference

Observed and inferred edges may share a broad endpoint relation when they assert the same biology, but provenance must preserve support mode and derivation. An inferred gene-expression assertion from a protein-product observation must not be mislabeled as an RNA measurement.


In [ ]:
# Distinguish observed assertions from controlled inference: run the bounded example and inspect its typed output.
semantics = pd.DataFrame([
    ("observed", "source record directly asserts relation", "evidence/"),
    ("inferred", "reviewed mapping or rule derives relation", "evidence_inferred/ or proof/"),
    ("context", "useful signal without accepted topology", "features/ or metadata/"),
], columns=["status", "meaning", "surface"])
display(semantics)


### Interpretation

Inference must fail closed when endpoint mapping, direction, sign, or context is unknown or conflicting. Coordinate overlap, co-mention, or generic association alone does not establish regulation or causality.


In [ ]:
forbidden_inferences = [
    "association score → causal direction",
    "RNA expression → observed protein expression",
    "coordinate overlap → observed regulation",
    "unknown pair → biological negative",
]
print("Do not infer:\n- " + "\n- ".join(forbidden_inferences))


### Checkpoint

When interpretation depends on an inference, locate its evidence or proof policy before using it as topology or supervision.


## Record provenance and interpretation boundaries

A reproducible answer records mode, root, relation, exact IDs, row limits, source releases, and helper version. This context allows another scientist to reproduce the bounded inspection and understand what was not queried.


In [ ]:
# Record provenance and interpretation boundaries: run the bounded example and inspect its typed output.
provenance_record = {
    "mode": MODE,
    "root": str(KG_ROOT),
    "relation": "disease_associated_gene",
    "query_gene": "ENSG00000141510",
    "edge_limit": 10,
    "evidence_limit": 50,
    "read_only": True,
}
print(json.dumps(provenance_record, indent=2))


### Interpretation

What this means: selected source records support represented graph assertions. What this does not mean: evidence sources are independent, coverage is complete, the association is causal, or intervention is effective and safe.


In [ ]:
print({
    "next_for_catalog_queries": "04_lamindb_equivalent_queries.ipynb",
    "next_for_graph_tensors": "05_sampled_pyg_heterodata.ipynb",
    "status_vocabulary": "bounded fixture inspection",
})


### Checkpoint

Use Notebook 04 to compare the canonical Parquet data plane with the currently partial LaminDB catalog mirror.
